# Regressão Logística para Classificação de Sentimentos

Nesta etapa, os embeddings gerados pelo modelo
`all-MiniLM-L6-v2` serão utilizados como características
de entrada para um modelo de Regressão Logística.

## Objetivo

Classificar as avaliações de filmes em duas classes:

- `0` = negativo
- `1` = positivo

## Etapas

1. Carregamento dos embeddings.
2. Verificação dos dados.
3. Separação entre características (`X`) e rótulos (`y`).
4. Divisão dos dados em treinamento e teste.
5. Treinamento da Regressão Logística.
6. Predição.
7. Avaliação do modelo.
8. Análise da matriz de confusão.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.append(str(PROJECT_ROOT / "src"))

from models.logistic_regression import (
    INPUT_FILE,
    load_data,
    prepare_data,
    split_data,
    create_model,
    train_model,
    evaluate_model,
)

In [ ]:
print("Arquivo de entrada:")
print(INPUT_FILE)

In [ ]:
df = load_data(INPUT_FILE)

print("Quantidade de linhas:", len(df))
print("Quantidade de colunas:", len(df.columns))

df.head()

In [ ]:
print("Distribuição dos sentimentos:")
print(df["sentiment"].value_counts().sort_index())

In [ ]:
embedding_columns = [
    column
    for column in df.columns
    if column.startswith("emb_")
]

print("Quantidade de embeddings:", len(embedding_columns))
print("Primeiras colunas:")
print(embedding_columns[:10])
print("Últimas colunas:")
print(embedding_columns[-10:])

In [ ]:
X, y = prepare_data(df)

print("Formato de X:", X.shape)
print("Formato de y:", y.shape)

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test = split_data(X, y)

print("Treinamento:", X_train.shape)
print("Validação:", X_val.shape)
print("Teste:", X_test.shape)

In [ ]:
print("Distribuição no treinamento:")
print(y_train.value_counts().sort_index())

print("\nDistribuição na validação:")
print(y_val.value_counts().sort_index())

print("\nDistribuição no teste:")
print(y_test.value_counts().sort_index())

In [ ]:
import numpy as np

C_VALUES = np.logspace(-2, 2, 13)

print(C_VALUES)

In [ ]:
results = []

for C in C_VALUES:
    model = create_model(C=C)

    train_model(
        model,
        X_train,
        y_train,
    )

    metrics, y_pred, y_proba = evaluate_model(
        model,
        X_val,
        y_val,
    )

    results.append(
        {
            "C": C,
            "accuracy": metrics["accuracy"],
            "precision": metrics["precision"],
            "recall": metrics["recall"],
            "f1": metrics["f1"],
        }
    )

validation_results = pd.DataFrame(results)

validation_results

In [ ]:
best_row = validation_results.loc[
    validation_results["f1"].idxmax()
]

best_C = best_row["C"]

print("Melhor configuração segundo a validação:")
print(f"C = {best_C}")
print(f"F1-score = {best_row['f1']:.4f}")

In [ ]:
X_train_final = pd.concat(
    [X_train, X_val],
    axis=0,
)

y_train_final = pd.concat(
    [y_train, y_val],
    axis=0,
)

print("Dados finais de treinamento:", X_train_final.shape)
print("Rótulos finais:", y_train_final.shape)

In [ ]:
final_model = create_model(C=float(best_C))

print(final_model)

In [ ]:
train_model(
    final_model,
    X_train_final,
    y_train_final,
)

print("Modelo final treinado.")

In [ ]:
test_metrics, y_test_pred, y_test_proba = evaluate_model(
    final_model,
    X_test,
    y_test,
)

In [ ]:
print("=== RESULTADO FINAL NO CONJUNTO DE TESTE ===")

print(
    f"Acurácia:  {test_metrics['accuracy']:.4f}"
)

print(
    f"Precisão:  {test_metrics['precision']:.4f}"
)

print(
    f"Recall:    {test_metrics['recall']:.4f}"
)

print(
    f"F1-score:  {test_metrics['f1']:.4f}"
)

In [ ]:
print("=== RELATÓRIO DE CLASSIFICAÇÃO ===")
print(
    test_metrics["classification_report"]
)

In [ ]:
print("=== MATRIZ DE CONFUSÃO ===")
print(
    test_metrics["confusion_matrix"]
)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay(
    confusion_matrix=test_metrics["confusion_matrix"],
    display_labels=["Negativo (0)", "Positivo (1)"],
).plot()

plt.title("Matriz de Confusão - Regressão Logística")
plt.show()

In [ ]:
comparison = pd.DataFrame(
    {
        "real": y_test.values,
        "previsto": y_test_pred,
    }
)

comparison.head(20)

In [ ]:
probabilities = pd.DataFrame(
    {
        "real": y_test.values,
        "previsto": y_test_pred,
        "probabilidade_negativo": y_test_proba[:, 0],
        "probabilidade_positivo": y_test_proba[:, 1],
    }
)

probabilities.head(10)

## Conclusão

Foi implementado um modelo de Regressão Logística utilizando os embeddings
gerados pelo modelo `all-MiniLM-L6-v2` como características de entrada.

Os dados foram divididos em três conjuntos:

- 70% para treinamento;
- 15% para validação;
- 15% para teste.

O conjunto de validação foi utilizado para comparar diferentes valores
do parâmetro `C` da Regressão Logística.

Após a escolha da configuração com base na validação, o modelo final foi
treinado utilizando os dados de treinamento e validação e avaliado uma
única vez no conjunto de teste.

A avaliação foi realizada utilizando acurácia, precisão, recall, F1-score
e matriz de confusão.